# Exploratory data analysis: AI4I 2020 machine failures

Goal: understand the data before modeling. What does the table contain, how
imbalanced is the target, which columns must not be used as features, and which
signals separate failures from normal operation?

Run `uv run mf download` once to download the dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from sklearn.metrics import average_precision_score

from machine_failure.data import (
    FAILURE_MODE_COLUMNS,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    load_raw,
)

DATA_PATH = Path("..") / "data" / "raw" / "ai4i2020.csv"
df = load_raw(DATA_PATH)

In [ ]:
# Two categorical colors (validated for colorblind separation), plus neutral ink.
NO_FAILURE, FAILURE = "#2a78d6", "#eb6834"
SURFACE, INK, INK_MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
CLASS_COLORS = {0: NO_FAILURE, 1: FAILURE}

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": GRID,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "text.color": INK,
        "axes.labelcolor": INK_MUTED,
        "xtick.color": INK_MUTED,
        "ytick.color": INK_MUTED,
        "axes.titlelocation": "left",
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "legend.frameon": False,
        "figure.dpi": 100,
    }
)
class_legend = [
    Patch(color=NO_FAILURE, label="No failure"),
    Patch(color=FAILURE, label="Failure"),
]

## 1. Shape and quality

In [ ]:
print(f"rows: {len(df):,}   columns: {df.shape[1]}")
print(f"missing values: {int(df.isna().sum().sum())}")
print(f"duplicate rows: {int(df.duplicated().sum())}")
print(f"UID unique: {df['UID'].is_unique}")
print(f"Product ID unique: {df['Product ID'].is_unique}")
print(f"Product ID starts with Type: {(df['Product ID'].str[0] == df['Type']).all()}")
df.dtypes.rename("dtype").to_frame()

In [ ]:
df[NUMERIC_FEATURES].describe().T

`UID` is just the row number and `Product ID` repeats the `Type` letter plus a
serial number, so both identifiers carry no signal and are dropped later.

## 2. Target: class balance

In [ ]:
counts = df[TARGET_COLUMN].value_counts().sort_index()
rate = df[TARGET_COLUMN].mean()

fig, ax = plt.subplots(figsize=(5.5, 3.5))
bars = ax.bar(
    ["No failure", "Failure"], counts.to_numpy(), color=[NO_FAILURE, FAILURE], width=0.5
)
for bar, n in zip(bars, counts.to_numpy(), strict=True):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        n,
        f"{n:,} ({n / len(df):.1%})",
        ha="center",
        va="bottom",
        color=INK,
    )
ax.set_ylabel("Machines")
ax.set_ylim(0, counts.max() * 1.12)
ax.set_title(f"Only {rate:.1%} of machines fail")
ax.grid(axis="x", visible=False)
plt.show()
print(f"imbalance: about {counts[0] / counts[1]:.1f} to 1")

A model that always predicts "no failure" scores about 96.6% accuracy, so
accuracy is useless here. We will use PR-AUC, recall, precision and F1.

## 3. Failure modes and leakage

In [ ]:
mode_counts = df[FAILURE_MODE_COLUMNS].sum().sort_values()

fig, ax = plt.subplots(figsize=(5.5, 3.2))
bars = ax.barh(mode_counts.index, mode_counts.to_numpy(), color=FAILURE, height=0.6)
for bar, n in zip(bars, mode_counts.to_numpy(), strict=True):
    ax.text(n + 1, bar.get_y() + bar.get_height() / 2, f"{n}", va="center")
ax.set_xlabel("Rows with the flag set")
ax.set_title("Heat dissipation and overstrain are the most common modes")
ax.grid(axis="y", visible=False)
plt.show()

In [ ]:
any_flag = (df[FAILURE_MODE_COLUMNS].sum(axis=1) > 0).rename("any mode flag")
pd.crosstab(df[TARGET_COLUMN], any_flag)

In [ ]:
rnf = df[df["RNF"] == 1]
n_rnf_failures = int(rnf[TARGET_COLUMN].sum())
print(f"RNF rows: {len(rnf)}, of which Machine failure = 1: {n_rnf_failures}")
n_multi = int((df[FAILURE_MODE_COLUMNS].sum(axis=1) > 1).sum())
print(f"rows with more than one flag: {n_multi}")

The flags describe *why* a failure happened, so they are known only after the
failure. Using them as features would leak the target and must be avoided. They
also do not match the target exactly: 9 failures have no flag, and 18 of the 19
`RNF` rows are not failures. That residual noise puts a ceiling on how well the
target can be predicted.

## 4. Feature distributions by class

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, col in zip(axes.flat, NUMERIC_FEATURES, strict=False):
    sns.histplot(
        data=df,
        x=col,
        hue=TARGET_COLUMN,
        palette=CLASS_COLORS,
        stat="density",
        common_norm=False,
        element="step",
        bins=40,
        ax=ax,
        legend=False,
    )
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("Density")
axes.flat[-1].axis("off")
axes.flat[-1].legend(handles=class_legend, loc="center", fontsize=11)
fig.suptitle("Each histogram is normalized within its class", x=0.01, ha="left")
fig.tight_layout()
plt.show()

In [ ]:
df.groupby(TARGET_COLUMN)[NUMERIC_FEATURES].median().T.rename(
    columns={0: "median (no failure)", 1: "median (failure)"}
)

## 5. Product type and tool wear

In [ ]:
by_type = df.groupby("Type")[TARGET_COLUMN].agg(["count", "mean"]).loc[["L", "M", "H"]]

fig, ax = plt.subplots(figsize=(5.5, 3.5))
bars = ax.bar(by_type.index, by_type["mean"] * 100, color=FAILURE, width=0.5)
for bar, (rate_pct, n) in zip(
    bars, zip(by_type["mean"] * 100, by_type["count"], strict=True), strict=True
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        rate_pct,
        f"{rate_pct:.1f}%\nn={n:,}",
        ha="center",
        va="bottom",
    )
ax.set_ylabel("Failure rate (%)")
ax.set_xlabel("Product quality type")
ax.set_ylim(0, by_type["mean"].max() * 100 * 1.3)
ax.set_title("Failure rate falls from L to M to H")
ax.grid(axis="x", visible=False)
plt.show()

In [ ]:
bins = [-1, 50, 100, 150, 200, 260]
labels = ["0-50", "51-100", "101-150", "151-200", "201+"]
wear_bucket = pd.cut(df["Tool wear"], bins=bins, labels=labels)
by_wear = df.groupby(wear_bucket, observed=True)[TARGET_COLUMN].agg(["count", "mean"])

fig, ax = plt.subplots(figsize=(6.5, 3.5))
bars = ax.bar(
    by_wear.index.astype(str), by_wear["mean"] * 100, color=FAILURE, width=0.6
)
for bar, rate_pct in zip(bars, by_wear["mean"] * 100, strict=True):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        rate_pct,
        f"{rate_pct:.1f}%",
        ha="center",
        va="bottom",
    )
ax.set_ylabel("Failure rate (%)")
ax.set_xlabel("Tool wear (minutes)")
ax.set_ylim(0, by_wear["mean"].max() * 100 * 1.2)
ax.set_title("Failure rate jumps once tool wear passes 200 minutes")
ax.grid(axis="x", visible=False)
plt.show()
by_wear

## 6. Relationships between features

In [ ]:
corr = df[[*NUMERIC_FEATURES, TARGET_COLUMN]].corr()
diverging = LinearSegmentedColormap.from_list(
    "diverging", [NO_FAILURE, "#f0efec", FAILURE]
)

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap=diverging,
    vmin=-1,
    vmax=1,
    linewidths=2,
    linecolor=SURFACE,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax,
)
ax.set_title("Temperatures move together; torque and speed move in opposition")
ax.grid(False)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
normal = df[df[TARGET_COLUMN] == 0]
failed = df[df[TARGET_COLUMN] == 1]
ax.scatter(
    normal["Rotational speed"], normal["Torque"], s=8, alpha=0.25, color=NO_FAILURE
)
ax.scatter(
    failed["Rotational speed"],
    failed["Torque"],
    s=16,
    alpha=0.8,
    color=FAILURE,
    edgecolor=SURFACE,
    linewidth=0.5,
)
ax.set_xlabel("Rotational speed (rpm)")
ax.set_ylabel("Torque (Nm)")
ax.set_title("Failures sit at the extremes of the torque-speed curve")
ax.legend(handles=class_legend, loc="upper right")
plt.show()

## 7. Derived features and the physical failure rules

The dataset documentation describes the failure modes with simple rules on
combinations of the raw features. Preview of the derived quantities (they are
implemented properly in the feature-engineering step):

- `temp_diff` = process temperature - air temperature (K)
- `power` = torque x rotational speed (W)
- `strain` = tool wear x torque (minNm)

In [ ]:
derived = df.assign(
    temp_diff=df["Process temperature"] - df["Air temperature"],
    power=df["Torque"] * df["Rotational speed"] * 2 * np.pi / 60,
    strain=df["Tool wear"] * df["Torque"],
)
DERIVED = ["temp_diff", "power", "strain"]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, col in zip(axes, DERIVED, strict=True):
    sns.histplot(
        data=derived,
        x=col,
        hue=TARGET_COLUMN,
        palette=CLASS_COLORS,
        stat="density",
        common_norm=False,
        element="step",
        bins=40,
        ax=ax,
        legend=False,
    )
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("Density")
fig.legend(handles=class_legend, loc="lower center", ncol=2)
fig.tight_layout(rect=(0, 0.07, 1, 1))
plt.show()

In [ ]:
y = df[TARGET_COLUMN]
scores = {}
for col in [*NUMERIC_FEATURES, *DERIVED]:
    s = derived[col]
    scores[col] = max(average_precision_score(y, s), average_precision_score(y, -s))
single_feature = pd.Series(scores, name="PR-AUC").sort_values(ascending=False)
print(f"base rate (a random score would reach about this): {y.mean():.3f}")
single_feature.round(3).to_frame()

In [ ]:
type_threshold = df["Type"].map({"L": 11000, "M": 12000, "H": 13000})
rules = {
    "HDF": (derived["temp_diff"] < 8.6) & (df["Rotational speed"] < 1380),
    "PWF": (derived["power"] < 3500) | (derived["power"] > 9000),
    "OSF": derived["strain"] > type_threshold,
}
for name, rule in rules.items():
    print(
        f"{name}: documented rule reproduces the flag exactly: "
        f"{bool((rule.astype(int) == df[name]).all())}"
    )

rule_hit = pd.concat(rules, axis=1).any(axis=1)
caught = int((rule_hit & (y == 1)).sum())
false_alarms = int((rule_hit & (y == 0)).sum())
print(f"failures caught by the three rules: {caught} of {int(y.sum())}")
print(f"false alarms from the three rules: {false_alarms}")

## 8. Takeaways for modeling

- **Data quality:** 10,000 rows, no missing values, no duplicates, unique
  identifiers. Nothing to clean.
- **Imbalance:** 339 failures (3.39%). Use stratified splits, cross-validation,
  and PR-AUC, recall and precision instead of accuracy. A 20% test set holds only
  about 68 failures, so a single test score is noisy.
- **Features:** use `Type` and the five measurements. Drop `UID`, `Product ID` and
  the five failure-mode flags, which leak the target.
- **Raw features are weak alone.** Failures show higher median torque and tool
  wear and lower median speed, but the distributions overlap heavily and
  correlations with the target stay small (torque is the highest, at 0.19).
- **Failures sit at both extremes.** The failure histograms are bimodal: torque
  has a small group of failures near 10 Nm as well as the main group at 50-70 Nm,
  and speed has failures at both the lowest and the highest rpm. A straight-line
  relationship cannot capture that, which is another reason to prefer tree-based
  models.
- **Combinations matter.** `power` and `strain` are stronger single signals than
  any raw feature, and three documented rules on them reproduce the `HDF`, `PWF`
  and `OSF` flags exactly. Tree-based models with these derived features should
  work much better than linear models on raw columns.
- **There is a ceiling.** Tool wear failures are random within 200-240 minutes,
  `RNF` is random, and 9 failures have no recorded cause. Perfect recall is not
  achievable, and a very high score would mean the model rediscovered the rules,
  not that it generalizes to real machines. The data is synthetic.